In [1]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python path: {sys.path}")

Python executable: c:\Users\Anupam\Desktop\LLM\.venv\Scripts\python.exe
Python path: ['C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10\\python310.zip', 'C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10\\DLLs', 'C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10\\lib', 'C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv', '', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages\\win32', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages\\win32\\lib', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages\\Pythonwin']


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.9.1+cu126
True


In [30]:
import os
import torch
from trl.trainer.sft_trainer import SFTTrainer
from dotenv import load_dotenv
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments


In [2]:
load_dotenv()  # Load environment variables from .env file
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN") or ""

In [33]:
# Disable torch.compile globally
torch._dynamo.config.disable = True
os.environ["TORCH_COMPILE_DISABLE"] = "1"

Mlflow implementation

In [3]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")  # Set the tracking URI to your MLflow server
mlflow.set_experiment("LLM_Fine_Tuning")  # Set the experiment name

[urllib3.connectionpool|WARNING]Retrying (Retry(total=6, connect=6, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it")': /api/2.0/mlflow/experiments/get-by-name?experiment_name=LLM_Fine_Tuning
[urllib3.connectionpool|WARNING]Retrying (Retry(total=5, connect=5, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it")': /api/2.0/mlflow/experiments/get-by-name?experiment_name=LLM_Fine_Tuning


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1773153482475, experiment_id='1', last_update_time=1773153482475, lifecycle_stage='active', name='LLM_Fine_Tuning', tags={}, workspace='default'>

Load the base model and tokenizer

In [4]:
max_seq_length = 2048
model_name= "unsloth/Qwen3.5-2B"
model, processor = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit = False,     # MoE QLoRA not recommended, dense 27B is fine
    load_in_16bit = True,     # bf16/16-bit LoRA
    full_finetuning = False,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 617/617 [00:06<00:00, 96.16it/s, Materializing param=model.visual.pos_embed.weight]                                 


In [17]:
type(processor.tokenizer)
print(processor.tokenizer)
# CRITICAL: Extract the text tokenizer from the vision processor
tokenizer = processor.tokenizer  # This gives you the text-only tokenizer
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"Set pad_token to: {tokenizer.pad_token}")
print(f"Vocab size: {len(tokenizer)}")


TokenizersBackend(name_or_path='unsloth/Qwen3.5-2B', vocab_size=248044, model_max_length=262144, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|vision_pad|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}, added_tokens_decoder={
	248044: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248045: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248046: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248047: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248048: AddedToken("<|object_ref_end|>"

In [18]:
tokenizer.tokenize

<bound method TokenizersBackend.tokenize of TokenizersBackend(name_or_path='unsloth/Qwen3.5-2B', vocab_size=248044, model_max_length=262144, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|im_end|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}, added_tokens_decoder={
	248044: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248045: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248046: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248047: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),


Apply LoRA adapters for efficient fine-tuning

In [7]:
# Enable peft for memory efficient training
# Disable vision fine-tuning - only train text layers
# This effectively treats it as a text-only model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",  # Text layers only
    ],
    lora_alpha = 16,
    # without dropout and without bias, to speed up training.
    lora_dropout = 0,
    bias = "none",
    # "unsloth" checkpointing is intended for very long context + lower VRAM
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = max_seq_length,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


Load and format the training dataset

In [8]:
# Load the dataset from huggingface
dataset = load_dataset("nohurry/Opus-4.6-Reasoning-3000x-filtered", split="train")
print(f"Original columns name:{dataset.column_names}")
# Keep only the columns you want (e.g., 'conversations' and 'text')
columns_to_keep = ['problem','thinking', 'solution']  # Replace with your actual column names
dataset = dataset.select_columns(columns_to_keep)
print(f"New columns: {dataset.column_names}")
# dataset = dataset.train_test_split(test_size=0.1, seed=3407)

Original columns name:['id', 'problem', 'thinking', 'solution', 'difficulty', 'category', 'timestamp', 'hash']
New columns: ['problem', 'thinking', 'solution']


In [9]:
type(dataset[0])
dataset[0]

{'problem': "252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?",
 'thinking': "Let me work through this problem step by step.\n\nFirst, I need to understand what's being asked: 252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?\n\nKey values given: 252, 8, 41, 300,000, 7,500, ,\n\nMy approach:\n1. Find the total number of people going on the field trip\n\n- Fifth-grade students: 252\n- Teachers: 8\n- Total people: 252 + 8 = 260 people\n2. Calculate how many buses are needed\n\nEach bus has 41 seats.\n\n$$\\text{Number of buses} = \\frac{260}{41} = 6.34...$$\n\nSince we cannot rent a partial bus, we must round up to the\

In [10]:
# 2. Format into conversations
def format_reasoning_conversation(examples):
    conversations = []
    for i in range(len(examples['problem'])):
        conversations.append([
            {"role": "user", "content": examples['problem'][i]},
            {"role": "assistant", "content": f"Let me think through this step-by-step:\n\n{examples['thinking'][i]}\n\nTherefore, the solution is:\n\n{examples['solution'][i]}"}
        ])
    return {"conversations": conversations}

print(f"Original dataset before the mapping:{dataset}")
dataset = dataset.map(format_reasoning_conversation, batched=True, remove_columns=dataset.column_names)
print(f"New dataset columns after the removing of previous columns:{dataset}")

Original dataset before the mapping:Dataset({
    features: ['problem', 'thinking', 'solution'],
    num_rows: 2326
})
New dataset columns after the removing of previous columns:Dataset({
    features: ['conversations'],
    num_rows: 2326
})


In [11]:
dataset["conversations"][0]

[{'content': "252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?",
  'role': 'user'},
 {'content': "Let me think through this step-by-step:\n\nLet me work through this problem step by step.\n\nFirst, I need to understand what's being asked: 252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?\n\nKey values given: 252, 8, 41, 300,000, 7,500, ,\n\nMy approach:\n1. Find the total number of people going on the field trip\n\n- Fifth-grade students: 252\n- Teachers: 8\n- Total people: 252 + 8 = 260 people\n2. Calculate how many buses are needed\n\nEach bus has 41 seats.\n\n$$\\text{Number of buses} = \\frac{260}{41} = 6.34...$$\

Apply the Qwen3.5:2b chat template

In [ ]:
# For the ChatML format
chat_template= """
<|im_start|>user
{user}<|im_end|>
<|im_start|>assistant
{OUTPUT}<|im_end|>"""

In [19]:
from unsloth import get_chat_template, standardize_sharegpt
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml"
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False,add_generation_prompt=False) for convo in convos
    ]
    return {
        "text": texts
    }

Model does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [20]:
print(tokenizer.pad_token)

<|PAD_TOKEN|>


In [21]:
dataset

Dataset({
    features: ['conversations'],
    num_rows: 2326
})

In [22]:
dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)

Map: 100%|██████████| 2326/2326 [00:00<00:00, 13909.49 examples/s]


In [23]:
print(type(dataset))
print(dataset["text"][0])
print(dataset.column_names)

<class 'datasets.arrow_dataset.Dataset'>
<|im_start|>user
252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?<|im_end|>
<|im_start|>assistant
Let me think through this step-by-step:

Let me work through this problem step by step.

First, I need to understand what's being asked: 252 fifth-grade students and 8 teachers at Yeji's school are going on a field trip. If the cost of renting a 41-seater bus is 300,000 won and the highway toll per bus is 7,500 won, how much does it cost to rent the bus and pay the toll?

Key values given: 252, 8, 41, 300,000, 7,500, ,

My approach:
1. Find the total number of people going on the field trip

- Fifth-grade students: 252
- Teachers: 8
- Total people: 252 + 8 = 260 people
2. Calculate how many buses are needed

Each bus has 41 seats.

$$\text{Number of buses} = \frac{

Standardize the conversation format(Optional)

In [ ]:
# from unsloth.chat_templates import standardize_sharegpt
# dataset = standardize_sharegpt(dataset)

In [24]:
# from unsloth import is_bfloat16_supported
from trl.trainer.sft_config import SFTConfig

In [25]:
# Define SFT-specific configuration
sft_config = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "mlflow",     # For Weights and Biases
        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "text",
        packing=False, # Must be False for raw text
        # dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    )

In [28]:
# # Training arguments remain the same
# training_args = TrainingArguments(
#     learning_rate=3e-4,
#     lr_scheduler_type="linear",
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     num_train_epochs=3,
#     fp16=not is_bfloat16_supported(),
#     bf16 = is_bfloat16_supported(),
#     logging_steps=1,
#     optim="adamw_8bit",
#     weight_decay=0.01,
#     warmup_steps=10,
#     output_dir="output",
#     report_to="mlflow",  # This should work with comet_ml installed [citation:4][citation:8]
#     seed=0,
# )

In [26]:
print(tokenizer.eos_token)
print(f"Tokenizer vocab size: {len(tokenizer)}")

<|im_end|>
Tokenizer vocab size: 248078


In [31]:
# Initialize SFTTrainer with proper parameter placement
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,  # Correct for v0.22.0 [citation:5]
    train_dataset=dataset,
    args=sft_config,            # Pass SFTConfig here [citation:9]
)

Unsloth: Tokenizing ["text"]: 100%|██████████| 2326/2326 [00:03<00:00, 708.24 examples/s]


In [34]:
import mlflow 
with mlflow.start_run(run_name="Qwen3.5:2b Pretraining", log_system_metrics=True):
    # Log key hyperparameter explicitly
    mlflow.log_params({
        "model_name": model_name,
        "max_seq_length": max_seq_length,
        "lora_r": 16,
        "lora_alpha": 16,
        "lora_dropout": 0,
        "lora_bias": "none",
        "target_modules": "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",
        "train_size": len(dataset),
        # "eval_size": len(dataset["test"]),
    }
    )
    trainer.train()

2026/03/13 02:15:10 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/13 02:15:10 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,326 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)


Step,Training Loss
1,1.318461
2,1.143638
3,1.670209
4,1.041791
5,1.047796
6,0.758909
7,1.391103
8,1.018917
9,0.835542
10,0.881359


2026/03/13 03:02:13 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/13 03:02:13 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run Qwen3.5:2b Pretraining at: http://localhost:5000/#/experiments/1/runs/512337f997ea405b96ef076295b7961b
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [35]:
model.push_to_hub("Anupammaiti/qwen_lora", token = os.getenv("HF_TOKEN")) # Online saving
tokenizer.push_to_hub("Anupammaiti/qwen_lora", token = os.getenv("HF_TOKEN")) # Online saving

Processing Files (1 / 1): 100%|██████████| 43.7MB / 43.7MB, 4.36MB/s  
New Data Upload: 100%|██████████| 43.7MB / 43.7MB, 4.36MB/s  


Saved model to https://huggingface.co/Anupammaiti/qwen_lora


Processing Files (1 / 1): 100%|██████████| 20.0MB / 20.0MB, 4.54MB/s  
New Data Upload: 100%|██████████| 20.0MB / 20.0MB, 4.54MB/s  
